# E3 — Regressione e incertezza

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emanbuc/corso_foundation_tabular/blob/main/notebooks/usare-tfm/E3_regressione_incertezza.ipynb)

**Corso 1 · Usare un TFM — Modulo 2 (laboratorio, 15 min)**

Un TFM per la regressione non restituisce solo un numero: restituisce una distribuzione, da cui si leggono media, mediana e quantili.

**Idea da far emergere:** l'intervallo predittivo si allarga dove mancano dati; una stima puntuale e una distribuzione non sono la stessa cosa.

- **Dati:** una sinusoide 1D con rumore eteroschedastico e un "buco" nei dati; poi California Housing (campione da 5.000 righe).
- **Modello:** `TabICLRegressor`.

Serve la GPU per la seconda parte; la prima gira anche su CPU.

## 0. Setup

L'API per quantili e intervalli dipende dalla versione di `tabicl`: qui è fissata alla 2.2.0, in cui `predict` accetta `output_type` e `alphas`.

In [ ]:
%pip install -q tabicl==2.2.0

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tabicl import TabICLRegressor

print("GPU disponibile:", torch.cuda.is_available())

## 1. Una sinusoide con un buco

Il rumore cresce con `x` (eteroschedastico) e tra 4 e 6 non c'è nessun dato.

In [ ]:
def deviazione_rumore(x):
    return 0.05 + 0.04 * x


def genera_sinusoide(n=300, buco=(4.0, 6.0), seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 10, 5 * n)
    x = x[(x < buco[0]) | (x > buco[1])][:n]
    y = np.sin(x) + rng.normal(0, deviazione_rumore(x))
    return x.reshape(-1, 1), y


BUCO = (4.0, 6.0)
X, y = genera_sinusoide(buco=BUCO)

plt.figure(figsize=(7, 3.5))
plt.scatter(X[:, 0], y, s=8, color="gray")
plt.axvspan(*BUCO, color="orange", alpha=0.15, label="nessun dato")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## 2. Dalla stima puntuale alla distribuzione

`predict` restituisce di default la media. Con `output_type` si chiedono anche i quantili della distribuzione predittiva, ai livelli indicati in `alphas`.

In [ ]:
ALPHAS = [0.05, 0.25, 0.5, 0.75, 0.95]

reg = TabICLRegressor()
reg.fit(X, y)

griglia = np.linspace(0, 10, 400).reshape(-1, 1)
pred = reg.predict(griglia, output_type=["mean", "quantiles"], alphas=ALPHAS)
media, quantili = pred["mean"], pred["quantiles"]
print("media:", media.shape, " quantili:", quantili.shape)

In [ ]:
g = griglia[:, 0]
plt.figure(figsize=(8, 4))
plt.scatter(X[:, 0], y, s=8, color="gray", label="dati")
plt.plot(g, np.sin(g), "k--", linewidth=1, label="funzione vera")
plt.plot(g, media, color="C0", label="media predetta")
plt.fill_between(g, quantili[:, 0], quantili[:, 4], color="C0", alpha=0.15, label="intervallo al 90%")
plt.fill_between(g, quantili[:, 1], quantili[:, 3], color="C0", alpha=0.30, label="intervallo al 50%")
plt.axvspan(*BUCO, color="orange", alpha=0.15)
plt.xlabel("x")
plt.ylabel("y")
plt.legend(loc="lower left", fontsize=8)
plt.show()

**Grafico da commentare:** l'ampiezza dell'intervallo al 90% lungo `x`, confrontata con quella del rumore vero.

Ci sono due motivi diversi per cui l'intervallo può essere largo: i dati sono rumorosi, oppure i dati mancano.

In [ ]:
ampiezza = quantili[:, 4] - quantili[:, 0]
ampiezza_rumore = 2 * 1.645 * deviazione_rumore(g)   # intervallo al 90% di una gaussiana

plt.figure(figsize=(8, 3.5))
plt.plot(g, ampiezza, label="intervallo al 90% predetto")
plt.plot(g, ampiezza_rumore, "k--", linewidth=1, label="intervallo al 90% del solo rumore")
plt.axvspan(*BUCO, color="orange", alpha=0.15, label="nessun dato")
plt.xlabel("x")
plt.ylabel("ampiezza")
plt.legend(fontsize=8)
plt.show()

## 3. Un dataset reale: California Housing

Si predice il valore mediano delle case di un distretto (in centinaia di migliaia di dollari) da 8 variabili. Contesto: 5.000 righe. Test: 1.000 righe.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

case = fetch_california_housing(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    case.data, case.target, train_size=5000, test_size=1000, random_state=0)
y_te = y_te.to_numpy()
print("Contesto:", X_tr.shape, " Test:", X_te.shape)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score

inizio = time.perf_counter()
reg = TabICLRegressor()
reg.fit(X_tr, y_tr)
pred = reg.predict(X_te, output_type=["mean", "quantiles"], alphas=[0.05, 0.5, 0.95])
print(f"Tempo TFM: {time.perf_counter() - inizio:.1f} s")

media = pred["mean"]
q05, mediana, q95 = pred["quantiles"].T

boosting = HistGradientBoostingRegressor(random_state=0).fit(X_tr, y_tr)

print(f"R² del TFM (media):         {r2_score(y_te, media):.3f}")
print(f"R² del gradient boosting:   {r2_score(y_te, boosting.predict(X_te)):.3f}")

### L'intervallo dice la verità?

Un intervallo al 90% è utile solo se contiene il valore vero circa nel 90% dei casi. Questa frazione si chiama **copertura**.

In [ ]:
dentro = (y_te >= q05) & (y_te <= q95)
print(f"Copertura dell'intervallo al 90%: {dentro.mean():.1%}")

ordine = np.argsort(mediana)[::10]   # una casa ogni 10, ordinate per valore predetto
plt.figure(figsize=(8, 4))
plt.errorbar(range(len(ordine)), mediana[ordine],
             yerr=[mediana[ordine] - q05[ordine], q95[ordine] - mediana[ordine]],
             fmt="none", ecolor="C0", alpha=0.5, label="intervallo al 90%")
plt.scatter(range(len(ordine)), y_te[ordine], s=10, c=np.where(dentro[ordine], "k", "red"),
            label="valore vero (rosso = fuori intervallo)", zorder=3)
plt.xlabel("distretti del test, ordinati per mediana predetta")
plt.ylabel("valore mediano delle case")
plt.legend(fontsize=8)
plt.show()

### Intervallo largo, errore grande?

Si dividono i distretti del test in tre gruppi per ampiezza dell'intervallo e si guarda l'errore medio in ciascun gruppo.

In [ ]:
tabella = pd.DataFrame({
    "ampiezza": q95 - q05,
    "errore assoluto": np.abs(y_te - mediana),
    "dentro l'intervallo": dentro,
})
tabella["gruppo"] = pd.qcut(tabella["ampiezza"].rank(method="first"), 3, labels=["stretto", "medio", "largo"])
tabella.groupby("gruppo", observed=True).mean().round(3)

## Da osservare

- Sulla sinusoide, l'intervallo si allarga dentro il buco? E la media, lì dentro, resta plausibile?
- A destra del buco l'intervallo è più largo che a sinistra? Se sì, la causa è il rumore dei dati, non la loro assenza.
- Su California Housing, i distretti con l'intervallo più largo sono anche quelli con l'errore più grande?

Domande per la discussione:

- In quale decisione concreta usereste il quantile al 5% invece della media?
- Se la copertura misurata fosse 75% invece di 90%, che cosa ne concludereste?

## Esercizio

Spostare il "buco" ai bordi del dominio e commentare l'estrapolazione.

Suggerimenti:

- generare i dati con `genera_sinusoide(buco=(8.0, 10.0))`, così mancano i dati a destra;
- predire su una griglia che va oltre il dominio, per esempio da 0 a 12;
- guardare sia la media sia l'ampiezza dell'intervallo: il modello "sa" di essere fuori dai dati?

In [ ]:
# TODO: X_b, y_b = genera_sinusoide(buco=(8.0, 10.0))

# TODO: fit e predict su una griglia da 0 a 12, con media e quantili

# TODO: ridisegnare i due grafici della sezione 2